[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# Dataclasses


## What you will be able to do

Replace the `__init__`, `__repr__` and `__eq__` you have been writing by hand with `@dataclass`,
decide which fields take part in comparing and printing, check values when an object is built, and
make objects that cannot be changed.


## The idea

### The problem

Every class in this guide has begun with the same chores. `__init__` takes the name, the readings
and the unit, and copies each one onto `self`: the names written twice that the **Your First Class**
notebook explained. Then comes a `__repr__`, from **Dunder Methods**, naming every field again so
the object prints readably. Then an `__eq__`, comparing every field again, and a `__hash__` if the
objects are to go in a set.

For a class with three fields, that is eleven lines before it does anything of its own, and each
field is written in four places. Adding a field means editing all four, and no error is raised when
one is missed. Forget the `__eq__`, and two objects that differ in the new field compare equal,
silently, in a program that looked finished.

These chores are the same for every class. The only thing that changes from one class to the next
is the list of fields, so Python can write them, if it is told the list.

### What a dataclass is

> A **dataclass** is a class whose fields are declared once, as names with type annotations in the
> class body, such as `name: str`. The `@dataclass` decorator reads that list and writes `__init__`,
> `__repr__` and `__eq__` from it. Adding a field to the list adds it to all three.

### Why it works that way

`@dataclass` is a decorator applied to a class, which the **Decorators** notebook said was possible
because a class is a value too. It takes the class, finds every annotated name in its body, writes
the methods, and hands the same class back.

The annotation is what makes a name a field. `unit: str = "C"` is a field with a default, while
`UNITS = ("C", "F")`, with no annotation, stays a class attribute as it was in **Class and Static
Methods**. The annotation's type is not checked, though. `name: str` documents what belongs there,
and Python will store a number in it without a word, which is the quiet error at the end.

The decorator also knows several of the mistakes this guide has collected. It refuses a list as a
default, the shared-default bug from **Your First Class**, and names what to use instead. It removes
`__hash__` because it wrote `__eq__`, the rule from **Dunder Methods**. And it takes options:
`frozen=True` for objects that cannot change, which can then be hashed, and `order=True` for objects
that sort.

A dataclass is still an ordinary class, and methods, class methods and checks can all be added to
it.

### Where you will meet this

Records read from files and web services are the commonest use: each row or response becomes an
object with named fields that prints readably and compares by value. The **JSON on Disk** notebook's
round trip becomes two calls, and the `from_dict` constructor from **Class and Static Methods** is
not needed at all when the JSON keys match the field names.

### What this notebook covers

The same class written by hand and as a dataclass, through the same numbered steps. Then what the
decorator wrote, defaults and `default_factory`, checks in `__post_init__`, choosing which fields
compare and print, `frozen=True`, `order=True`, JSON, and methods on a dataclass. Then a station and
its readings, written from two lists of fields.

### A first look

A dataclass with two fields. There is nothing to run yet: read it, and read the output underneath
it.

```python
from dataclasses import dataclass


@dataclass
class Station:
    name: str
    readings: list[float]


print(Station("Tromso", [-4.1, -2.6]))
print(Station("Tromso", [-4.1, -2.6]) == Station("Tromso", [-4.1, -2.6]))
```

```
Station(name='Tromso', readings=[-4.1, -2.6])
True
```

No `__init__`, `__repr__` or `__eq__` was written, and all three work.


## Setup

Three imports, the first bringing in six names from the `dataclasses` module.

- `dataclass` is the decorator this notebook is about
- `field` gives one field its own settings: a default built fresh for each object, and whether the
  field is compared or printed
- `fields` lists the fields a dataclass has, to show what the decorator found
- `asdict` turns a dataclass object into a dictionary, for JSON
- `replace` makes a changed copy of an object that cannot be changed
- `FrozenInstanceError` is what an unchangeable object raises when something assigns to it
- `inspect` shows the signature of the `__init__` the decorator wrote
- `json` writes a dataclass out as JSON and reads it back, in one section

**Run this cell before the rest of the notebook.**


In [1]:
from dataclasses import dataclass, field, fields, asdict, replace, FrozenInstanceError
import inspect
import json

print("ready")


ready


## Worked examples

### Before and after: written by hand, or written from a list of fields

Here is the problem from the top of this notebook, in code. Each version goes through the same
numbered steps:

1. Build a station and print it.
2. Compare two stations with the same fields. They should be equal.
3. Compare two that differ only in unit. They should not be equal.

Then a field is added, and one more step checks it:

4. Compare two stations that differ only in elevation. They should not be equal.

First, the class written by hand, as every class in this guide has been so far.


In [2]:
class Station:
    def __init__(self, name, readings, unit="C"):
        self.name = name
        self.readings = readings
        self.unit = unit

    def __repr__(self):
        return f"Station(name={self.name!r}, readings={self.readings!r}, unit={self.unit!r})"

    def __eq__(self, other):
        if not isinstance(other, Station):
            return NotImplemented
        return (self.name, self.readings, self.unit) == (other.name, other.readings, other.unit)


Steps 1 to 3.


In [3]:
# 1. Build a station and print it.
north = Station("Tromso", [-4.1, -2.6])
print("1.", north)

# 2. Compare two stations with the same fields. They should be equal.
print("2.", Station("Tromso", [-4.1, -2.6]) == Station("Tromso", [-4.1, -2.6]))

# 3. Compare two that differ only in unit. They should not be equal.
print("3.", Station("Tromso", [-4.1, -2.6]) == Station("Tromso", [-4.1, -2.6], "F"))


1. Station(name='Tromso', readings=[-4.1, -2.6], unit='C')
2. True
3. False


All three answers are right. Count what it took: eleven lines, not counting the blank ones, and each
of the three fields written in four places: the parameter, the assignment, the `__repr__` and the
`__eq__`.

Now the same class as a dataclass.


In [4]:
@dataclass
class Station:
    name: str
    readings: list[float]
    unit: str = "C"


The same three steps, with the code unchanged.


In [5]:
# 1. Build a station and print it.
north = Station("Tromso", [-4.1, -2.6])
print("1.", north)

# 2. Compare two stations with the same fields. They should be equal.
print("2.", Station("Tromso", [-4.1, -2.6]) == Station("Tromso", [-4.1, -2.6]))

# 3. Compare two that differ only in unit. They should not be equal.
print("3.", Station("Tromso", [-4.1, -2.6]) == Station("Tromso", [-4.1, -2.6], "F"))


1. Station(name='Tromso', readings=[-4.1, -2.6], unit='C')
2. True
3. False


The same three answers and the same printout, from a class whose body is its list of fields. The
decorator wrote the `__init__`, the `__repr__` and the `__eq__` that the hand-written version spelled
out.

Now the change that shows why that matters. A new field arrives, `elevation`. In the hand-written
class it has to be added to `__init__`, to `__repr__` and to `__eq__`. Here is that edit with `__eq__`
missed, which is the easy one to miss, because nothing uses it until two stations are compared.


In [6]:
class Station:
    def __init__(self, name, readings, unit="C", elevation=0):
        self.name = name
        self.readings = readings
        self.unit = unit
        self.elevation = elevation

    def __repr__(self):
        return (f"Station(name={self.name!r}, readings={self.readings!r}, "
                f"unit={self.unit!r}, elevation={self.elevation!r})")

    def __eq__(self, other):
        if not isinstance(other, Station):
            return NotImplemented
        return (self.name, self.readings, self.unit) == (other.name, other.readings, other.unit)


Step 4.


In [7]:
# 4. Compare two stations that differ only in elevation. They should not be equal.
low = Station("Tromso", [-4.1, -2.6], elevation=10)
high = Station("Tromso", [-4.1, -2.6], elevation=100)
print("4.", low)
print("  ", high)
print("   equal:", low == high)


4. Station(name='Tromso', readings=[-4.1, -2.6], unit='C', elevation=10)
   Station(name='Tromso', readings=[-4.1, -2.6], unit='C', elevation=100)
   equal: True


The two printouts show different elevations, and `==` says the stations are equal, because `__eq__`
still compares only the first three fields. Nothing raised. A program that removed duplicate stations
would now throw one of these away.

In the dataclass, the new field is one line.


In [8]:
@dataclass
class Station:
    name: str
    readings: list[float]
    unit: str = "C"
    elevation: int = 0


Step 4 again, with the code unchanged.


In [9]:
# 4. Compare two stations that differ only in elevation. They should not be equal.
low = Station("Tromso", [-4.1, -2.6], elevation=10)
high = Station("Tromso", [-4.1, -2.6], elevation=100)
print("4.", low)
print("  ", high)
print("   equal:", low == high)


4. Station(name='Tromso', readings=[-4.1, -2.6], unit='C', elevation=10)
   Station(name='Tromso', readings=[-4.1, -2.6], unit='C', elevation=100)
   equal: False


`equal: False`. The field went into the list once, and the decorator put it into all three methods.

| | Written by hand | `@dataclass` |
|---|---|---|
| Lines for three fields | 11 | 5 |
| Places each field is written | 4 | 1 |
| Adding `elevation` | edit `__init__`, `__repr__` and `__eq__` | add one line |
| Step 4, after the edit | equal, wrongly | not equal |

The rest of this notebook takes the dataclass apart.

| Question about the dataclass version | The section that answers it |
|---|---|
| What exactly did the decorator write? | What the decorator wrote |
| How does a field get a list as its default? | Defaults, and `default_factory` |
| Where does a check go, with no `__init__` to put it in? | Checking values in `__post_init__` |
| Can a field be left out of `==`, or out of the printout? | Choosing which fields compare and print |
| How do I make objects that cannot change? | `frozen=True` |

### What the decorator wrote

`@dataclass` above `class Station` means `Station = dataclass(Station)`, by the rule from the
**Decorators** notebook. The decorator found the annotated names, wrote the methods into the class,
and handed it back.


In [10]:
print("the __init__ it wrote:", inspect.signature(Station))
print("the fields it found:  ", [f.name for f in fields(Station)])
print("methods it wrote:     ", [name for name in ("__init__", "__repr__", "__eq__") if name in vars(Station)])
print("__hash__ is None:     ", Station.__hash__ is None)


the __init__ it wrote: (name: str, readings: list[float], unit: str = 'C', elevation: int = 0) -> None
the fields it found:   ['name', 'readings', 'unit', 'elevation']
methods it wrote:      ['__init__', '__repr__', '__eq__']
__hash__ is None:      True


The signature is the `__init__` you would have written by hand, with the defaults in place. The last
line is the rule from **Dunder Methods** applied for you: a class that defines `__eq__` loses its
default `__hash__`, so these stations cannot go in a set. `frozen=True`, below, is how a dataclass gets
one back.

### Defaults, and `default_factory`

A field with a default is written like an assignment with an annotation. A name assigned without an
annotation is not a field at all.


In [11]:
@dataclass
class Station:
    UNITS = ("C", "F")
    name: str
    unit: str = "C"


print("fields:       ", [f.name for f in fields(Station)])
print("Station.UNITS:", Station.UNITS)
print(Station("Tromso"))


fields:        ['name', 'unit']
Station.UNITS: ('C', 'F')
Station(name='Tromso', unit='C')


`UNITS` has no annotation, so it stayed a class attribute, shared by every station, and the decorator
left it out of `__init__`, `__repr__` and `__eq__`.

A list as a default is refused.


In [12]:
try:
    @dataclass
    class Refused:
        name: str
        readings: list[float] = []
except ValueError as error:
    print("ValueError:", error)


ValueError: mutable default <class 'list'> for field readings is not allowed: use default_factory


That is the shared default from the **Your First Class** notebook, which a hand-written `__init__`
accepts without a word. The dataclass refuses it at the moment the class is defined, and names the
fix.


In [13]:
@dataclass
class Station:
    name: str
    readings: list[float] = field(default_factory=list)


tromso = Station("Tromso")
malaga = Station("Malaga")
tromso.readings.append(-4.1)

print("Tromso:", tromso.readings)
print("Malaga:", malaga.readings)
print("one list:", tromso.readings is malaga.readings)


Tromso: [-4.1]
Malaga: []
one list: False


`field(default_factory=list)` gives the field a function to call instead of a value to share, and the
generated `__init__` calls it once for every object. Each station gets its own new list.

### Checking values in `__post_init__`

There is no `__init__` to put a check in, because the decorator writes it. `__post_init__` is the
place instead: if the class defines one, the generated `__init__` calls it last, after every field is
stored.


In [14]:
@dataclass
class Station:
    UNITS = ("C", "F")
    name: str
    unit: str = "C"

    def __post_init__(self):
        self.name = self.name.strip()
        if self.unit not in self.UNITS:
            raise ValueError(f"unit must be one of {self.UNITS}, not {self.unit!r}")


print(Station("  Tromso  "))

try:
    Station("Oslo", "K")
except ValueError as error:
    print("refused:", error)

later = Station("Bodo")
later.unit = "K"
print("a later assignment is not checked:", later)


Station(name='Tromso', unit='C')
refused: unit must be one of ('C', 'F'), not 'K'
a later assignment is not checked: Station(name='Bodo', unit='K')


`__post_init__` tidied the name and refused a bad unit, when each station was built. The last line
shows its limit: it runs once, at construction, and assigning to `unit` afterwards runs nothing. A
property from the **Properties** notebook would check every assignment, and `frozen=True`, below,
forbids them altogether.

### Choosing which fields compare and print

`field` also decides whether a field takes part in `==` and in the printout. A free-text note is a
good example: it belongs to the station, and it should not make two records of the same station
unequal, or clutter every printout.


In [15]:
@dataclass
class Station:
    name: str
    unit: str = "C"
    note: str = field(default="", compare=False, repr=False)


with_note = Station("Tromso", note="sensor replaced in March")
without_note = Station("Tromso")

print(with_note)
print("equal:", with_note == without_note)
print("the note is still there:", repr(with_note.note))


Station(name='Tromso', unit='C')
equal: True
the note is still there: 'sensor replaced in March'


The note is stored and can be read, but `__repr__` leaves it out and `__eq__` ignores it. The
**Dunder Methods** notebook's exercise compared books on title and author and ignored page count;
`compare=False` is how a dataclass says the same thing.

### `frozen=True`: objects that cannot change

A frozen dataclass refuses every assignment after it is built, and because its fields can no longer
change, the decorator can safely write a `__hash__` from them.


In [16]:
@dataclass(frozen=True)
class Reading:
    when: str
    celsius: float


first = Reading("2026-03-01", -4.1)

try:
    first.celsius = 99.0
except FrozenInstanceError as error:
    print("FrozenInstanceError:", error)

print("in a set:", {first, Reading("2026-03-01", -4.1)})
print("replace: ", replace(first, celsius=-5.0))
print("original:", first)


FrozenInstanceError: cannot assign to field 'celsius'
in a set: {Reading(when='2026-03-01', celsius=-4.1)}
replace:  Reading(when='2026-03-01', celsius=-5.0)
original: Reading(when='2026-03-01', celsius=-4.1)


Assignment is refused, two equal readings collapse to one in a set, and `replace` builds a changed
copy while leaving the original alone, which is the style of `in_fahrenheit` from the **Methods**
notebook.

Frozen stops a field from being reassigned. It does not stop the object held in the field from
changing.


In [17]:
@dataclass(frozen=True)
class FrozenStation:
    name: str
    readings: list[float]


shallow = FrozenStation("Tromso", [-4.1])
shallow.readings.append(999.0)
print("appended to a frozen station:", shallow)

try:
    hash(shallow)
except TypeError as error:
    print("hash:", error)


appended to a frozen station: FrozenStation(name='Tromso', readings=[-4.1, 999.0])
hash: unhashable type: 'list'


The list itself was never frozen, so `append` changed it, and it also made the object impossible to
hash. For values that must not change, store a tuple, which cannot be changed and can be hashed.

### `order=True`: objects that sort

`order=True` writes `<` and the other comparisons, comparing field by field in the order the fields
are listed, as tuples compare.


In [18]:
@dataclass(frozen=True, order=True)
class Reading:
    when: str
    celsius: float


out_of_order = [
    Reading("2026-03-03", -3.8),
    Reading("2026-03-01", -4.1),
    Reading("2026-03-02", -2.6),
]

for reading in sorted(out_of_order):
    print(reading)
print("latest:", max(out_of_order))


Reading(when='2026-03-01', celsius=-4.1)
Reading(when='2026-03-02', celsius=-2.6)
Reading(when='2026-03-03', celsius=-3.8)
latest: Reading(when='2026-03-03', celsius=-3.8)


`when` is listed first, so it decides the order, and dates written as ISO strings sort as text in
time order. Put the field that should decide first at the top of the list.

### To JSON and back

`asdict` turns an object into a dictionary of its fields, and `json.dumps` writes that as text.
Coming back, `**` in a call is the other half of `**` in the **Functions** notebook, where
`def describe(**fields)` gathered keyword arguments into a dictionary. In a call it does the reverse,
spreading a dictionary out into keyword arguments, so `Station(**{"name": "Tromso"})` means
`Station(name="Tromso")`.


In [19]:
@dataclass
class Station:
    name: str
    readings: list[float]
    unit: str = "C"


north = Station("Tromso", [-4.1, -2.6])

text = json.dumps(asdict(north))
print("as JSON:", text)

back = Station(**json.loads(text))
print("read back:", back)
print("the same: ", back == north)


as JSON: {"name": "Tromso", "readings": [-4.1, -2.6], "unit": "C"}
read back: Station(name='Tromso', readings=[-4.1, -2.6], unit='C')
the same:  True


Because the JSON keys are the field names, no `from_dict` constructor was needed. The round trip
from **JSON on Disk** is now two calls.

### A dataclass is still a class

Methods, class methods and everything else from this guide go into a dataclass exactly as they would
into any class.


In [20]:
@dataclass
class Station:
    name: str
    readings: list[float] = field(default_factory=list)

    def mean(self):
        return round(sum(self.readings) / len(self.readings), 2)

    @classmethod
    def from_csv(cls, line):
        name, *values = line.split(",")
        return cls(name, [float(v) for v in values])


north = Station.from_csv("Tromso,-4.1,-2.6")
print(north, "has a mean of", north.mean())


Station(name='Tromso', readings=[-4.1, -2.6]) has a mean of -3.35


### Putting it together: a station and its readings

Two dataclasses, each written from its list of fields. `Reading` is frozen and orderable, so
readings can go in a set and sort into time order, and it refuses an impossible temperature in
`__post_init__`. `Station` holds its readings in a list built by `default_factory`, keeps them out of
its printout, checks its unit against a class attribute, and has ordinary methods and a class method.


In [21]:
@dataclass(frozen=True, order=True)
class Reading:
    """One reading. Dates are ISO strings, so text order is time order."""

    when: str
    celsius: float

    def __post_init__(self):
        if not -90 <= self.celsius <= 60:
            raise ValueError(f"{self.celsius} is not a temperature that occurs on Earth")


@dataclass
class Station:
    """A station and its readings, written from a list of fields."""

    UNITS = ("C", "F")
    name: str
    unit: str = "C"
    readings: list[Reading] = field(default_factory=list, repr=False)

    def __post_init__(self):
        self.name = self.name.strip()
        if self.unit not in self.UNITS:
            raise ValueError(f"unit must be one of {self.UNITS}, not {self.unit!r}")

    def record(self, when, celsius):
        self.readings.append(Reading(when, celsius))

    def coldest(self):
        return min(self.readings, key=lambda reading: reading.celsius)

    def history(self):
        return sorted(set(self.readings))

    @classmethod
    def from_csv(cls, line):
        name, unit = line.split(",")
        return cls(name, unit)


north = Station.from_csv("  Tromso  ,C")
north.record("2026-03-03", -3.8)
north.record("2026-03-01", -4.1)
north.record("2026-03-02", -2.6)
north.record("2026-03-01", -4.1)

print(north)
print("readings recorded:", len(north.readings))
for reading in north.history():
    print("  ", reading)
print("coldest:", north.coldest())


Station(name='Tromso', unit='C')
readings recorded: 4
   Reading(when='2026-03-01', celsius=-4.1)
   Reading(when='2026-03-02', celsius=-2.6)
   Reading(when='2026-03-03', celsius=-3.8)
coldest: Reading(when='2026-03-01', celsius=-4.1)


Four readings were recorded, one of them twice. `history` put them in a set, where the two identical
frozen readings became one, and `sorted` put the rest in time order because `when` is the first field.
The station printed without its readings, because that field has `repr=False`.

Now the checks, and the whole station as JSON.


In [22]:
try:
    north.record("2026-03-04", 999)
except ValueError as error:
    print("refused a reading:", error)

try:
    Station("Oslo", "K")
except ValueError as error:
    print("refused a station:", error)

print()
print(json.dumps(asdict(north)))


refused a reading: 999 is not a temperature that occurs on Earth
refused a station: unit must be one of ('C', 'F'), not 'K'

{"name": "Tromso", "unit": "C", "readings": [{"when": "2026-03-03", "celsius": -3.8}, {"when": "2026-03-01", "celsius": -4.1}, {"when": "2026-03-02", "celsius": -2.6}, {"when": "2026-03-01", "celsius": -4.1}]}


Both checks ran in `__post_init__`, one in each class. `asdict` went all the way down, turning each
`Reading` inside the station into a dictionary of its own.

### Where each part came from

| In the station and its readings | What it relies on | The section that showed it |
|---|---|---|
| no `__init__`, `__repr__` or `__eq__` written | the decorator writes them from the fields | Before and after |
| `readings` built by `default_factory` | a list default is refused, and a factory gives each object its own | Defaults, and `default_factory` |
| `UNITS` left out of the fields | a name without an annotation is a class attribute | Defaults, and `default_factory` |
| the name tidied, the unit and temperature checked | `__post_init__` runs after the fields are stored | Checking values in `__post_init__` |
| the station printing without its readings | `field(repr=False)` | Choosing which fields compare and print |
| duplicate readings removed by a set | `frozen=True` makes an object hashable | `frozen=True` |
| `history` in time order | `order=True` compares the first field first | `order=True` |
| the whole station as JSON | `asdict` goes all the way down | To JSON and back |
| `from_csv` on a dataclass | a dataclass is still a class | A dataclass is still a class |


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/11-dataclasses-solutions.ipynb).

**1.** Write `Book`, with `title`, `author` and `pages`, as a dataclass. Print one, then compare two
books with the same fields.


In [23]:
# your code here


**2.** Give `Book` a `tags` field, a list of strings that defaults to empty, written the way a
dataclass accepts. Add a tag to one book and show that another book's tags are unaffected.


In [24]:
# your code here


**3.** Add a `__post_init__` that raises `ValueError` when `pages` is less than 1, and show it refusing
a book of 0 pages.


In [25]:
# your code here


**4.** Make two books with the same title and author but different page counts compare equal, by
leaving `pages` out of the comparison.


In [26]:
# your code here


**5.** Make `Book` frozen, show that assigning to `pages` is refused, put two equal books in a set,
and use `replace` to make a copy with a different page count.


In [27]:
# your code here


**6.** Turn a book into JSON with `asdict`, read it back into a `Book` with `**`, and show that the
copy equals the original.


In [28]:
# your code here


## Common errors

### TypeError: no annotations, so no fields

The decorator finds fields by their annotations. Names assigned without one are class attributes, and
the generated `__init__` takes none of them.


In [29]:
@dataclass
class NoAnnotations:
    name = "unnamed"
    unit = "C"


print("fields:", [f.name for f in fields(NoAnnotations)])
NoAnnotations("Tromso", "F")


fields: []


TypeError: NoAnnotations.__init__() takes 1 positional argument but 3 were given

The class has no fields, so its `__init__` takes nothing but `self`, and the two arguments had nowhere
to go. `fields` returning an empty list is the quick way to see it. Write `name: str = "unnamed"`, with
the annotation, and the name becomes a field.

### TypeError: a field without a default after one with

Fields become the parameters of `__init__`, in order, and a parameter without a default cannot follow
one with a default, the rule from the **Functions** notebook.


In [30]:
@dataclass
class OutOfOrder:
    name: str
    unit: str = "C"
    readings: list[float]


TypeError: non-default argument 'readings' follows default argument 'unit'

This error arrives when the class is defined, not when an object is built, because that is when the
decorator writes `__init__`. Move the fields without defaults to the top, or give `readings` a default
of its own with `field(default_factory=list)`.

### TypeError: a plain dataclass in a set

A dataclass writes `__eq__`, so by the rule from **Dunder Methods** it has no `__hash__` unless it is
frozen.


In [31]:
@dataclass
class Station:
    name: str
    unit: str = "C"


{Station("Tromso")}


TypeError: cannot use 'Station' as a set element (unhashable type: 'Station')

`frozen=True` fixes it, and so does `@dataclass(eq=False)`, which leaves `__eq__` alone and keeps
comparison by identity. Choose frozen when the objects stand for values; the set then treats two
stations with the same fields as one.

### The quiet one: annotations are not checked

`name: str` and `readings: list[float]` look like rules. They are notes.


In [32]:
@dataclass
class Typed:
    name: str
    readings: list[float]


odd = Typed(42, "warm")

print(odd)
print("name is a", type(odd.name).__name__, "and readings is a", type(odd.readings).__name__)
print("len(odd.readings):", len(odd.readings))


Typed(name=42, readings='warm')
name is a int and readings is a str
len(odd.readings): 4


A number was stored as the name and a word as the readings, and nothing raised. `len` even gave an
answer, `4`, the number of letters in `warm`. The first error will come much later, from whatever
tries to add the readings up.

Python does not check annotations when a program runs. Separate tools called type checkers read them before a
program runs, and report values that do not match. Inside a dataclass, the check
that runs is the one you write in `__post_init__`.


## Recap

- `@dataclass` writes `__init__`, `__repr__` and `__eq__` from the list of annotated fields.
- Each field is written once. Adding one adds it to every generated method.
- A name without an annotation is a class attribute, not a field.
- Fields with defaults come after fields without them, as parameters do.
- A list as a default is refused. `field(default_factory=list)` gives each object its own.
- `__post_init__` runs after the fields are stored, and is where checks go.
- `__post_init__` checks once, at construction. Later assignments are not checked.
- `field(compare=False)` and `field(repr=False)` leave a field out of `==` or out of the printout.
- `frozen=True` refuses assignment and writes a `__hash__`, so objects can go in a set.
- Frozen is shallow: a list held in a frozen field can still change.
- `replace` makes a changed copy of a frozen object.
- `order=True` sorts by the fields in the order they are listed.
- `asdict` and `**` take a dataclass to JSON and back.
- Annotations are not checked when the program runs.


## What is next

The **Interfaces** notebook. The stations in this guide have all had a `report` method because each
class happened to define one. Nothing required it, and a class that forgot would fail only when
something finally called it. Abstract base classes and Protocols let code declare what it requires of
the objects it is given, and an abstract base class turns a missing method into an error the moment an
object is created.


---

&#8592; **Previous:** [Composition over Inheritance](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/10-composition-over-inheritance.ipynb)  &nbsp;·&nbsp;  [Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)
